# Hierarchical Clustering - UPGMA

---
## Learning Objectives

1. Implement a distance-based phylogeny neighbor joining algorithm
1. Understanding of hierarchical clustering
1. Manipulation of arrays and lists in Python


---
## Imports
You will need to install a two new packages to properly render your phylogenetic trees. Using your preferred package manager, install both [`biopython`](https://biopython.org/) and `matplotlib` into your virtual environments.

For example:
```bash
$ micromamba create -n module06 -c bioconda -c conda-forge python numpy matplotlib biopython
```

In [3]:
from typing import List, Tuple, Dict
from io import StringIO

import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np
import textdistance

---
## Background

Today we will implement a distance-based phylogenetic tree construction method using the Neighbor-Joining (NJ) algorithm and Smith-Waterman local alignment scores. Unlike UPGMA (Unweighted Pair Group Method using Arithmetic averages), NJ produces unrooted trees and does not assume a constant evolutionary rate across lineages, making it more biologically realistic for analyzing sequence relationships.

We will use HIV-1 reverse transcriptase sequences to construct our phylogenetic tree. The Smith-Waterman algorithm will be used to generate pairwise local alignment scores, which will then be converted into distances for tree construction. This approach is particularly suitable for HIV sequence analysis as it:
1. Handles sequence variations effectively through local alignment
2. Accounts for potential rate heterogeneity across different viral strains
3. Does not assume a molecular clock

The ultimate output will be an unrooted phylogenetic tree representing the evolutionary relationships between the HIV-1 sequences, visualized using the ete3 library. This method provides insights into viral diversity and evolutionary patterns while avoiding the assumptions of simpler hierarchical clustering approaches.

The key innovations of this implementation are:
- Use of Smith-Waterman for sensitive local alignment scoring
- Implementation of Neighbor-Joining for unrooted tree construction
- More biologically realistic evolutionary model

## Neighbor-Joining Algorithm

**Input**: Distance matrix $D$ for $n$ sequences

**Initialization**:
- Let $n$ be the number of sequences
- Assign each sequence $i$ to its own leaf node
- Initialize tree $T$ with leaf nodes

**Iteration**:
While $n > 2$:
1. Calculate Q-matrix where:
   $Q(i,j) = (n-2)d(i,j) - \sum_{k=1}^n d(i,k) - \sum_{k=1}^n d(j,k)$

2. Find pair $(i,j)$ with minimum $Q(i,j)$

3. Calculate branch lengths:
   $d_i = \frac{d(i,j) + (r_i - r_j)/(n-2)}{2}$
   $d_j = d(i,j) - d_i$
   where $r_i = \sum_{k=1}^n d(i,k)$

4. Create new node $k$
   - Add branches from $k$ to $i$ and $j$ with lengths $d_i$ and $d_j$

5. Update distances to remaining nodes $x$:
   $d(k,x) = \frac{d(i,x) + d(j,x) - d(i,j)}{2}$

6. Remove nodes $i$ and $j$
7. Add node $k$ to active nodes
8. $n = n - 1$

**Termination**:
When $n = 2$ with remaining nodes $i$ and $j$:
- Add final branch between $i$ and $j$ with length $d(i,j)$
- Return unrooted tree $T$

---
## Distance metrics for comparing sequences

### Smith-Waterman
This was previously implemented. Feel free to import if you know how...or Copy & Paste.



In [4]:
def read_fasta(filename: str) -> Dict[str, str]:
    """Reads sequences from FASTA file.

    Args:
        filename (str): Path to FASTA file containing HIV RT sequences

    Returns:
        Dict[str, str]: Dictionary mapping sequence IDs to sequences

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> len(seqs) > 0
        True
    """

    sequences = {}
    current_id = None
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                current_id = line.split()[-1]   # ID = last word
                sequences[current_id] = []
            else:
                sequences[current_id].append(line.upper())
    return {seq_id: "".join(parts) for seq_id, parts in sequences.items()}
    pass

In [ ]:
def smith_waterman_normalized_distance(
    sequence1: str,
    sequence2: str,
    match_score: float = 1.0
) -> float:
    """Implements Smith-Waterman local alignment algorithm.

    Args:
        sequence1 (str): First HIV RT sequence
        sequence2 (str): Second HIV RT sequence

    Returns:
        float: Normalized similarity score between sequences

    Examples:
        >>> seq1 = "ACGT"
        >>> seq2 = "ACGA"
        >>> smith_waterman(seq1, seq2)
        0.25
    """
    distance = textdistance.smith_waterman.normalized_distance(sequence1, sequence2)
    return(round(distance,3))

In [54]:
def build_distance_matrix(sequences: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """Builds DISTANCE (not similiarity) matrix using Smith-Waterman scores.

    Args:
        sequences (Dict[str, str]): Dictionary of HIV RT sequences

    Returns:
        Tuple[np.ndarray, List[str]]: Distance matrix and list of sequence IDs

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> mat[0:3, 0:3]  # Show 3x3 slice of distance matrix
        array([[0.000, 0.004, 0.012],
              [0.004, 0.000, 0.012],
              [0.012, 0.012, 0.000]])
        >>> len(ids)  # Number of sequences
        19
    """
    """
    ids = []
    sequences = []

    ids.append(sequences.key)
    id.append(sequence.values)

    make np matrix of 0s of size sequences x sequences
    
    for i in range(len(sequences) - 1):
        seq1 = seqeunces[i]
        for j in range(len(sequences) - 1):
            seq2 = sequences[j]
            distance = sw(seq1, seq2)
            matrix[i][j] = distance 
            j = j + 1
        i = i + 1
    """
    # create data structures
    ids_list = []
    sequence_list = []
    remove_index_list = []

    # store ids and sequences in parallel lists
    for key, value in sequences.items():
        ids_list.append(key)
        sequence_list.append(value)

    # identifiy duplicates and remove them from data 
    for i in range(len(sequence_list)):
        seq = sequence_list[i]
        for j in range(i+1,len(sequence_list)):
            remaining_seq = sequence_list[j]
            if seq == remaining_seq:
                remove_index_list.append(j)

    for index in remove_index_list:
        del sequence_list[index]
        del ids_list[index]

    # get matrix dimensions
    sequence_number = len(sequence_list)

    print(sequence_number)

    # create matrix
    distance_matrix = np.zeros((sequence_number,sequence_number), float)
    
    # fill matrix with calculated text distances
    # for each row: grab sequence from sequence_list corresponding to that row  
    for i in range(sequence_number):
        seq1 = sequence_list[i]

        # for each column: compare column sequence to all other sequences and add value to matrix
        for j in range(sequence_number):
            seq2 = sequence_list[j]
            distance = smith_waterman_normalized_distance(seq1, seq2)
            distance_matrix[i, j] = distance 
            j = j + 1
        i = i + 1
    
    # return tuple
    return(distance_matrix, ids_list)

In [ ]:
def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> str:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18
    """
    """
    Initialize values for nodes
    Add up each row of matrix
    Score using D*
    join pair + limb measure
    update matrix
    """

    # Newick string
    newick = ""

    # Inititalize tree
    nodes = len(labels)

    # Iteration 
    while nodes > 3:

        # add each row of distance matrix, giving a 1 dimensional np array
        TD_array = sum(distance_matrix)
        
        # calculate D* matrix
        D_star_matrix = np.zeros((nodes, nodes), float)
        
        # for each row
        for i in range(nodes):
            # for each column: if not the same as row, calculate D* from TD array and distance matrix
            for j in range(nodes):
                if i != j:
                    distance_star = (nodes - 2) * distance_matrix[i,j] - TD_array[i] - TD_array[j]
                    D_star_matrix[i, j] = distance_star 
                j = j + 1
            i = i + 1

        # find the min in the D* matrix and extract the coordinates
        min_number = np.argmin(D_star_matrix)
        min_coords = np.unravel_index(min_number, (nodes,nodes))
        row, column = min_coords

        # calculate branch lengths
        branch_split = (TD_array[row] - TD_array[column])/(nodes - 2)
        length_i = 0.5 * distance_matrix[row,column] + branch_split
        length_j = distance_matrix[row, column] - length_i

        # calculate distances to new node
        u_array = np.array([])
        for i in range(nodes):
            if i != row and i != column:
                u_distance = 0.5 * (distance_matrix[i, row] + distance_matrix[i, column])
                u_array_column = np.append(u_array, u_distance)

        # delete joined samples from distance matrix
        distance_matrix = np.delete(distance_matrix, (row, column), 0)
        distance_matrix = np.delete(distance_matrix, (row, column), 1)

        # add new distances to matrix
        distance_matrix = np.insert(distance_matrix, 0, u_array_column, 1) # row of length nodes - 1

        u_array_row = np.insert(u_array, 0, 0) # column of length nodes
        distance_matrix = np.insert(distance_matrix, 0, u_array_row, 0)
        
        # delete joined samples from labels
        remove_row = labels[row]
        remove_column = labels[column]

        labels.remove(remove_row)
        labels.remove(remove_column)

        labels.insert(0,"U") # new label

        # create string
        newick = newick + f"({labels[row]}:{length_i}, {labels[column]}:{length_j}), "

        # recursive
        return(newick + neighbor_joining(distance_matrix, labels))
    
    else:
        return("the last 3")
    
    


In [16]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [ ]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = read_fasta("data/lafayette_SARS_RT.fasta")
    

    # Build distance matrix using Smith-Waterman

    dist_matrix, seq_ids = build_distance_matrix(sequences)



    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)

    # Plot tree
    plot_tree(tree)

In [77]:
large_width = 400
np.set_printoptions(linewidth=large_width)

sequences = read_fasta("data/lafayette_SARS_RT.fasta")

dist_matrix, seq_ids = build_distance_matrix(sequences)

tree = neighbor_joining(dist_matrix, seq_ids)

print(tree)

19
(DM3:0.0011470588235294086, JT1:0.00785294117647059), (U:-0.04115625, P14:0.04115625), (U:-0.007000000000000014, P1:0.007000000000000014), (U:0.02643750000000001, P11:-0.02643750000000001), (U:0.020548076923076895, P4:-0.020548076923076895), (U:0.0180703125, P2:-0.0180703125), (U:0.014846590909090886, P5:-0.014846590909090886), (U:0.014085156250000008, P7:-0.014085156250000008), (U:0.0313359375, P6:-0.0313359375), (U:0.02357006835937498, P9:-0.02357006835937498), (U:0.025829241071428578, P12:-0.025829241071428578), (U:0.021694091796874985, P15:-0.021694091796874985), (U:0.025042675781250003, P13:-0.025042675781250003), (U:0.011043060302734376, DM1:-0.011043060302734376), (U:0.013289632161458334, DM2:-0.013289632161458334), (U:0.022412918090820313, P8:-0.022412918090820313), the last 3
